# Ark+ 3D: Heterogeneous Label Learning on MedMNIST 3D

**Reference**: Ma et al. *Nature* 2025 — Ark+: A fully open AI foundation model for chest X-rays.

This notebook adapts the Ark+ cyclic pretraining strategy for **3D MedMNIST** datasets,
using a memory-efficient 2D Swin-Tiny-per-slice + Depth Transformer encoder.

### Hardware Target
- **GPU**: RTX 4060 8 GB VRAM
- **RAM**: 16 GB
- **Batch size**: 8 (reduce to 4 if OOM)

### Architecture
```
Input (B,1,D,28,28)
  └─► Upsample slices → (B*D, 3, 224, 224)
        └─► 2D Swin-Tiny (pretrained) → (B*D, 768)
              └─► SliceProjector MLP → (B, D, 768)
                    └─► DepthAggregator Transformer → (B, 768)
                          ├─► ProjectorMLP → (B, 1376)   [consistency loss]
                          └─► TaskHead_i   → (B, C_i)    [task loss]
```

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import sys
import os

# Add project root to path (assumes notebook is in ark_plus_3d/notebooks/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from ark_plus_3d.models.ark_plus import ArkPlus3D
from ark_plus_3d.data.dataset import build_datasets, get_weighted_sampler, NAME_TO_FLAG
from ark_plus_3d.training.engine import train_ark_plus
from ark_plus_3d.utils.helpers import set_seed, get_device, estimate_vram_mb
from ark_plus_3d.utils.metrics import plot_training_curves, print_final_summary
import yaml

print('All imports OK ✓')

In [ ]:
# ── Cell 2: Config ─────────────────────────────────────────────────────────
# Load dataset config
config_path = os.path.join(project_root, 'ark_plus_3d/configs/medmnist_3d.yaml')
with open(config_path) as f:
    config = yaml.safe_load(f)

print('Available datasets:', list(config.keys()))

# ── Experiment selection ────────────────────────────────────────────────────
# Quick test: 3 datasets  |  Full: all 6
DATASETS = ['OrganMNIST3D', 'NoduleMNIST3D', 'AdrenalMNIST3D']   # Quick test
# DATASETS = list(NAME_TO_FLAG.keys())                             # Full 6-dataset

# ── Hyperparameters (RTX 4060 optimised) ───────────────────────────────────
BATCH_SIZE        = 8       # Reduce to 4 if OOM
LR                = 1e-4
EPOCHS            = 10      # Quick test | use 200 for full training
WARMUP_EPOCHS     = 5
MOMENTUM_TEACHER  = 0.996   # EMA base momentum
EVAL_EVERY        = 2       # Validate every N epochs
DATA_ROOT         = os.path.join(project_root, 'data')
SAVE_DIR          = os.path.join(project_root, 'checkpoints')
EXP_NAME          = 'ark_3d_test'

DEVICE = get_device(verbose=True)
set_seed(42)

# VRAM budget check
estimate_vram_mb(batch_size=BATCH_SIZE)

In [ ]:
# ── Cell 3: Data Loaders ───────────────────────────────────────────────────
from torch.utils.data import DataLoader

train_loaders, val_loaders, test_loaders = [], [], []

for ds_name in DATASETS:
    flag = NAME_TO_FLAG[ds_name]
    print(f'Loading {ds_name} ({flag})...', end=' ')
    
    tr, vl, te = build_datasets(flag, data_root=DATA_ROOT)
    sampler = get_weighted_sampler(tr)
    
    train_loaders.append(DataLoader(
        tr, batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=2, pin_memory=True, drop_last=True
    ))
    val_loaders.append(DataLoader(
        vl, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    ))
    test_loaders.append(DataLoader(
        te, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    ))
    print(f'train={len(tr)}, val={len(vl)}, test={len(te)} ✓')

print(f'\nLoaded {len(DATASETS)} datasets successfully.')

In [ ]:
# ── Cell 4: Model ──────────────────────────────────────────────────────────
import torch

# Number of output classes per dataset
# Binary tasks: 2 output logits (consistent with multi-class interface)
num_classes = [
    max(len(config[ds]['diseases']), 2)   # floor at 2 for binary
    for ds in DATASETS
]
print(f'num_classes per dataset: {dict(zip(DATASETS, num_classes))}')

model = ArkPlus3D(
    num_classes_list=num_classes,
    swin_model='swin_tiny_patch4_window7_224',
    embed_dim=768,
    projector_dim=1376,
    pretrained=True,   # ImageNet pretrained Swin-Tiny
    dropout=0.1,
).to(DEVICE)

counts = model.param_count()
print(f'\nTotal params    : {counts["total_M"]:.1f} M')
print(f'Trainable params: {counts["trainable_M"]:.1f} M  (teacher frozen)')

# Sanity check forward pass
dummy = torch.randn(2, 1, 28, 28, 28, device=DEVICE)
with torch.no_grad():
    proj, logits = model(dummy, head_idx=0, mode='student')
print(f'\nForward pass OK ✓')
print(f'  proj:   {tuple(proj.shape)}')    # (2, 1376)
print(f'  logits: {tuple(logits.shape)}')  # (2, n_classes[0])

if DEVICE.type == 'cuda':
    used_gb = torch.cuda.memory_allocated() / 1024**3
    print(f'  VRAM after model init: {used_gb:.2f} GB')

In [ ]:
# ── Cell 5: Train ──────────────────────────────────────────────────────────
# NOTE: Swin downloads pretrained weights on first run (~110 MB).
# Data downloads on first run (~500 MB total for 3 datasets).

history = train_ark_plus(
    model=model,
    datasets_names=DATASETS,
    datasets_config=config,
    train_loaders=train_loaders,
    val_loaders=val_loaders,
    test_loaders=test_loaders,
    epochs=EPOCHS,
    lr=LR,
    warmup_epochs=WARMUP_EPOCHS,
    momentum_teacher=MOMENTUM_TEACHER,
    device=DEVICE,
    save_dir=SAVE_DIR,
    exp_name=EXP_NAME,
    eval_every=EVAL_EVERY,
)

In [ ]:
# ── Cell 6: Visualise ──────────────────────────────────────────────────────
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig = plot_training_curves(
    history,
    save_path=os.path.join(SAVE_DIR, f'{EXP_NAME}_curves.png')
)

# Print final test summary
n_classes_list = [max(len(config[ds]['diseases']), 2) for ds in DATASETS]
print_final_summary(history['test_metrics'], DATASETS, n_classes_list)

In [ ]:
# ── Cell 7 (optional): Full 6-dataset run ─────────────────────────────────
# Uncomment to run all 6 MedMNIST 3D datasets for full Ark+ replication.
# Estimated time: ~2h per 100 epochs on RTX 4060.

# DATASETS_FULL = list(NAME_TO_FLAG.keys())
# train_loaders_full, val_loaders_full, test_loaders_full = [], [], []
# for ds_name in DATASETS_FULL:
#     flag = NAME_TO_FLAG[ds_name]
#     tr, vl, te = build_datasets(flag, data_root=DATA_ROOT)
#     sampler = get_weighted_sampler(tr)
#     train_loaders_full.append(DataLoader(tr, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True))
#     val_loaders_full.append(DataLoader(vl, batch_size=BATCH_SIZE, num_workers=2))
#     test_loaders_full.append(DataLoader(te, batch_size=BATCH_SIZE, num_workers=2))
#
# num_classes_full = [max(len(config[ds]['diseases']), 2) for ds in DATASETS_FULL]
# model_full = ArkPlus3D(num_classes_list=num_classes_full, pretrained=True).to(DEVICE)
#
# history_full = train_ark_plus(
#     model=model_full,
#     datasets_names=DATASETS_FULL,
#     datasets_config=config,
#     train_loaders=train_loaders_full,
#     val_loaders=val_loaders_full,
#     test_loaders=test_loaders_full,
#     epochs=200,
#     lr=1e-4,
#     warmup_epochs=10,
#     eval_every=10,
#     device=DEVICE,
#     save_dir=SAVE_DIR,
#     exp_name='ark_3d_full_200ep',
# )